# Bottleneck Dimension Sweep — Hopf-GAE Sensitivity Analysis

Trains the Hopf-GAE with $d_z \in \{3, 4, 6, 8, 12\}$ holding all other
hyperparameters fixed at v6 values. Reports HC–MDD separation, intervention
effects at four scales, and circuit enrichment for each bottleneck width.

**Purpose:** Demonstrate that findings are robust to the bottleneck dimension.

# 1 — Setup

In [1]:
# =============================================================================
# S1 --- LIBRARIES AND CONFIGURATION
# =============================================================================

import warnings
warnings.filterwarnings('ignore', category=FutureWarning)

import numpy as np
import pandas as pd
import torch
from scipy import stats as sp_stats
from torch_geometric.loader import DataLoader as PyGDataLoader

# Project modules (v4 with all 8 improvements)
from config import *
from models import *
from utils import *

# ── v6 config: Hopf-GAE (denoising graph autoencoder) ─────────────────────
# KL removed after 5 versions of collapse. Denoising + dropout replace it.

# Reconstruction feature weights for 7 targets:
#   [a, omega, chisq, s_plv, s_mvar_in, s_mvar_out, plv_within]
# Physics features weighted higher; connectivity features at 1.0
RECON_FEATURE_WEIGHTS = torch.tensor([2.0, 1.0, 1.0, 0.5, 0.5, 0.5, 0.5])
N_FEATURES_OUT = 7

# Denoising autoencoder noise level
NOISE_SIGMA = 0.1  # std of Gaussian noise on encoder input during training

# Dropout on latent z before decoder
Z_DROPOUT = 0.3

# Graph-level loss weight (mean + std of a per graph)
LAMBDA_GRAPH = 0.1

# No beta / KL / free-bits — removed entirely
# Training is a flat autoencoder loop, no annealing schedule

# HC holdout for false positive rate
HC_HOLDOUT_FRAC = 0.15

# Permutation test params
N_PERMUTATIONS = 10000

seed_everything(SEED)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_V4.mkdir(parents=True, exist_ok=True)

log.info('Configuration loaded. PyTorch %s | CUDA: %s', torch.__version__, torch.cuda.is_available())
log.info('Paths: UKF=%s  PLV=%s  MVAR=%s  HC_MVAR=%s',
         MDD_UKF_CSV.exists(), PLV_RDS.exists(), MVAR_RDS.exists(), HC_MVAR_RDS.exists())

SWEEP_DIMS = [3, 4, 6, 8, 12]
log.info('Bottleneck sweep: dims=%s', SWEEP_DIMS)


/opt/anaconda3/lib/python3.11/site-packages/pandas/core/computation/expressions.py:22: UserWarning: Pandas requires version '2.10.2' or newer of 'numexpr' (version '2.8.7' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED
/opt/anaconda3/lib/python3.11/site-packages/pandas/core/arrays/masked.py:56: UserWarning: Pandas requires version '1.4.2' or newer of 'bottleneck' (version '1.3.7' currently installed).
  from pandas.core import (
[02:17:36] Configuration loaded. PyTorch 2.11.0 | CUDA: False
[02:17:36] Paths: UKF=True  PLV=True  MVAR=True  HC_MVAR=True
[02:17:36] Bottleneck sweep: dims=[3, 4, 6, 8, 12]


# 2 — Data Loading

In [2]:
# =============================================================================
# S2 --- DATA LOADING
# =============================================================================

data = load_all_data()
ukf_df = data['ukf_df']
group_df = data['group_df']
plv_all = data['plv_all']
mvar_all = data['mvar_all']
topo_df = data['topo_df']

[02:17:36] UKF: 8205 rows, 19 subjects
[02:17:36] PLV: 38 matrices
[02:17:36] MVAR: 38 matrices


# 3 — ROI Metadata

In [3]:
# =============================================================================
# S3 --- ROI METADATA
# =============================================================================

assert ukf_df is not None, 'UKF data not loaded'
roi_meta, network_assignment, N_NETWORKS = build_roi_meta_and_assignment(ukf_df)

[02:17:36] ROI metadata: 216 ROIs, 8 networks, 69 circuit ROIs


# 4 — Structural Connectivity

In [4]:
# =============================================================================
# S4 --- STRUCTURAL CONNECTIVITY
# =============================================================================

sc_matrix, roi_centroids = load_or_build_sc(roi_meta)
assert sc_matrix is not None, 'SC matrix construction failed'
log.info('SC matrix: shape %s, density %.2f%%',
         sc_matrix.shape, 100.0 * (sc_matrix > 0.001).sum() / sc_matrix.shape[0]**2)

[02:17:36] SC from atlas centroids: (216, 216)
[02:17:36] SC matrix: shape (216, 216), density 96.80%


# 5 — Simulator

In [5]:
# =============================================================================
# S5 --- SIMULATOR
# =============================================================================

simulator = StuartLandauSimulator(
    sc_matrix=sc_matrix, n_rois=sc_matrix.shape[0],
    TR=TR, n_TRs=N_VOLS, dt=0.1, sigma=0.02
)
log.info('Simulator ready: %d ROIs, %d TRs', sc_matrix.shape[0], N_VOLS)

[02:17:36] Simulator ready: 216 ROIs, 260 TRs


# 6 — QC and Test Graph

In [6]:
# =============================================================================
# S6 --- QC
# =============================================================================

log.info('=' * 70)
log.info('DATA QUALITY CONTROL')
log.info('=' * 70)

for session in ukf_df['session'].unique():
    s_df = ukf_df[ukf_df['session'] == session]
    log.info('  %s: n=%d, mean_a=%.4f +/- %.4f, subcritical=%.1f%%',
             session, len(s_df), s_df['a'].mean(), s_df['a'].std(),
             100 * (s_df['a'] < 0).mean())

test_subj = ukf_df['subject'].iloc[0]
test_sess = ukf_df['session'].iloc[0]
test_group = ukf_df[ukf_df['subject'] == test_subj]['group'].iloc[0]
test_plv = np.array(plv_all.get(f'{test_subj}|{test_sess}')) if plv_all else None
test_mvar = None
if mvar_all and f'{test_subj}|{test_sess}' in mvar_all:
    m = mvar_all[f'{test_subj}|{test_sess}']
    test_mvar = np.asarray(m) if hasattr(m, 'shape') else None

test_graph = build_subject_graph(
    subject=test_subj, session=test_sess, ukf_df=ukf_df, roi_meta=roi_meta,
    plv_mat=test_plv, mvar_mat=test_mvar, sc_mat=sc_matrix, group=test_group,
)
log.info('Test graph: %s, x=%s, a=[%.4f, %.4f]',
         test_subj, test_graph.x.shape,
         test_graph.x[:, 0].min().item(), test_graph.x[:, 0].max().item())
for rel in EDGE_RELATIONS:
    ei = f'edge_index_{rel}'
    if hasattr(test_graph, ei):
        log.info('  %s edges: %d', rel.upper(), getattr(test_graph, ei).shape[1])

roi_meta.to_csv(RESULTS_DIR / 'roi_meta_216.csv', index=False)
np.save(RESULTS_DIR / 'sc_matrix_216.npy', sc_matrix)
torch.save(network_assignment, RESULTS_DIR / 'network_assignment_216.pt')
log.info('Phase 1 artifacts saved.')

[02:17:36] ======================================================================
[02:17:36] DATA QUALITY CONTROL
[02:17:36] ======================================================================
[02:17:36]   rest1: n=4101, mean_a=-0.2854 +/- 0.1573, subcritical=100.0%
[02:17:36]   rest2: n=4104, mean_a=-0.2906 +/- 0.1620, subcritical=100.0%
[02:17:36] Test graph: AC_E5694, x=torch.Size([216, 11]), a=[-0.7213, -0.0542]
[02:17:36]   PLV edges: 4644
[02:17:36]   MVAR edges: 2337
[02:17:36]   SC edges: 7310
[02:17:36] Phase 1 artifacts saved.


# 7 — Encoder

In [7]:
# =============================================================================
# S7 --- HOPF ENCODER (v4: classification removed)
# =============================================================================

encoder = HopfEncoder(
    n_node_features=3 + N_NETWORKS,
    hidden_dim=HIDDEN_DIM,
    dropout=DROPOUT,
)

total_params = sum(p.numel() for p in encoder.parameters())
log.info('HopfEncoder: %d total parameters', total_params)
for name, module in encoder.named_children():
    n = sum(p.numel() for p in module.parameters())
    log.info('  %-20s %6d params', name, n)

encoder.eval()
with torch.no_grad():
    enc_out = encoder(test_graph)
log.info('Smoke test: h=%s, a_pred=%s',
         enc_out['node_embeddings'].shape, enc_out['a_pred'].shape)
log.info('Initial relation weights: %s',
         {k: f'{w:.3f}' for k, w in zip(RELATION_KEYS, encoder.conv1.relation_weights.tolist())})

[02:17:36] HopfEncoder: 5485 total parameters
[02:17:36]   conv1                  1286 params
[02:17:36]   conv2                  3302 params
[02:17:36]   input_proj              352 params
[02:17:36]   physics_head            545 params
[02:17:36] Smoke test: h=torch.Size([216, 32]), a_pred=torch.Size([216, 1])
[02:17:36] Initial relation weights: {'edge_index_plv': '0.333', 'edge_index_mvar': '0.333', 'edge_index_sc': '0.333'}


# 8 — Synthetic Data

In [8]:
# =============================================================================
# S8 --- SYNTHETIC DATASET GENERATION
# =============================================================================

log.info('=' * 70)
log.info('STAGE 1: SYNTHETIC DATA GENERATION')
log.info('=' * 70)

syn_graphs = []
for i in range(N_SYN + N_VAL_SYN):
    rng = np.random.RandomState(SEED + 1000 + i)
    a_mean = rng.uniform(-0.40, -0.05)
    a_std = rng.uniform(0.05, 0.20)
    G = rng.uniform(0.3, 1.5)

    sim_result = simulator.generate_graph(
        a_mean=a_mean, a_std=a_std, G=G,
        seed=SEED + 1000 + i, compute_connectivity=True,
    )

    n_rois = len(sim_result['a_true'])
    x_sim = torch.zeros(n_rois, 3 + N_NETWORKS, dtype=torch.float32)
    x_sim[:, 0] = torch.tensor(sim_result['a_true'], dtype=torch.float32)
    x_sim[:, 1] = torch.tensor(sim_result['omega'], dtype=torch.float32)
    x_sim[:, 2] = 0.5
    if network_assignment is not None:
        for j in range(n_rois):
            if j < len(network_assignment):
                x_sim[j, 3 + network_assignment[j].item()] = 1.0

    ei_plv, ea_plv = matrix_to_edge_index(sim_result['plv'], directed=False, top_k_pct=PLV_TOP_K)
    ei_mvar, ea_mvar = matrix_to_edge_index(sim_result['mvar'], directed=True, threshold=1e-10)
    ei_sc, ea_sc = matrix_to_edge_index(sc_matrix, directed=False, top_k_pct=SC_TOP_K)

    data = Data(
        x=x_sim,
        edge_index_plv=ei_plv, edge_attr_plv=ea_plv,
        edge_index_mvar=ei_mvar, edge_attr_mvar=ea_mvar,
        edge_index_sc=ei_sc, edge_attr_sc=ea_sc,
        a_true=torch.tensor(sim_result['a_true'], dtype=torch.float32),
        num_nodes=n_rois,
    )
    syn_graphs.append(data)
    if (i + 1) % 50 == 0:
        log.info('  Generated %d / %d', i + 1, N_SYN + N_VAL_SYN)

syn_train = syn_graphs[:N_SYN]
syn_val = syn_graphs[N_SYN:]
log.info('Synthetic dataset: %d train + %d val', len(syn_train), len(syn_val))

[02:17:36] ======================================================================
[02:17:36] STAGE 1: SYNTHETIC DATA GENERATION
[02:17:36] ======================================================================
[02:18:07]   Generated 50 / 220
[02:18:37]   Generated 100 / 220
[02:19:04]   Generated 150 / 220
[02:19:32]   Generated 200 / 220
[02:19:42] Synthetic dataset: 200 train + 20 val


# 9 — Pre-Training

In [9]:
# =============================================================================
# S9 --- PRE-TRAINING
# =============================================================================

log.info('=' * 70)
log.info('STAGE 1: PRE-TRAINING')
log.info('=' * 70)

train_loader = PyGDataLoader(syn_train, batch_size=PRETRAIN_BS, shuffle=True)
val_loader = PyGDataLoader(syn_val, batch_size=len(syn_val))

criterion = HopfPhysicsLoss(lambda_physics=LAMBDA_PHYSICS, lambda_subcrit=LAMBDA_SUBCRIT)
optimizer = torch.optim.AdamW(encoder.parameters(), lr=PRETRAIN_LR, weight_decay=PRETRAIN_WD)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=PRETRAIN_EPOCHS)

history = {'epoch': [], 'train_loss': [], 'val_loss': [], 'val_r2_mean_a': []}

for epoch in range(1, PRETRAIN_EPOCHS + 1):
    encoder.train()
    epoch_loss, n_batches = 0.0, 0
    for batch_data in train_loader:
        optimizer.zero_grad()
        result = encoder(batch_data)
        a_pred = result['a_pred']
        a_true = batch_data.a_true.unsqueeze(-1) if batch_data.a_true.dim() == 1 else batch_data.a_true
        loss, _ = criterion(a_pred=a_pred, a_true=a_true)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(encoder.parameters(), max_norm=1.0)
        optimizer.step()
        epoch_loss += loss.item()
        n_batches += 1
    scheduler.step()

    encoder.eval()
    with torch.no_grad():
        val_preds, val_true, val_loss = [], [], 0.0
        for vb in val_loader:
            r = encoder(vb)
            a_p = r['a_pred']
            a_t = vb.a_true.unsqueeze(-1) if vb.a_true.dim() == 1 else vb.a_true
            vl, _ = criterion(a_pred=a_p, a_true=a_t)
            val_loss += vl.item()
            for b in range(vb.batch.max().item() + 1):
                mask = (vb.batch == b)
                val_preds.append(a_p[mask].mean().item())
                val_true.append(a_t[mask].mean().item())
        val_preds, val_true = np.array(val_preds), np.array(val_true)
        ss_res = np.sum((val_preds - val_true) ** 2)
        ss_tot = np.sum((val_true - val_true.mean()) ** 2)
        r2 = 1 - ss_res / max(ss_tot, 1e-12)

    history['epoch'].append(epoch)
    history['train_loss'].append(epoch_loss / max(n_batches, 1))
    history['val_loss'].append(val_loss)
    history['val_r2_mean_a'].append(r2)

    if epoch % 20 == 0 or epoch == 1:
        log.info('  Epoch %3d | Train %.4f | Val %.4f | R2 %.3f | LR %.1e',
                 epoch, history['train_loss'][-1], val_loss, r2, scheduler.get_last_lr()[0])

log.info('Pre-training complete. R2 = %.3f', history['val_r2_mean_a'][-1])
log.info('Learned relation weights (conv1): %s',
         {k: f'{w:.3f}' for k, w in zip(RELATION_KEYS, encoder.conv1.relation_weights.tolist())})
log.info('Learned relation weights (conv2): %s',
         {k: f'{w:.3f}' for k, w in zip(RELATION_KEYS, encoder.conv2.relation_weights.tolist())})

pretrain_path = RESULTS_DIR / 'hopf_encoder_pretrained.pt'
torch.save(encoder.state_dict(), pretrain_path)
pd.DataFrame(history).to_csv(RESULTS_DIR / 'pretrain_history.csv', index=False)

[02:19:42] ======================================================================
[02:19:42] STAGE 1: PRE-TRAINING
[02:19:42] ======================================================================
[02:19:43]   Epoch   1 | Train 0.0266 | Val 0.0270 | R2 -0.010 | LR 3.0e-03
[02:19:58]   Epoch  20 | Train 0.0125 | Val 0.0125 | R2 0.962 | LR 2.7e-03
[02:20:13]   Epoch  40 | Train 0.0103 | Val 0.0100 | R2 0.977 | LR 2.0e-03
[02:20:28]   Epoch  60 | Train 0.0095 | Val 0.0099 | R2 0.962 | LR 1.0e-03
[02:20:44]   Epoch  80 | Train 0.0090 | Val 0.0091 | R2 0.979 | LR 2.9e-04
[02:20:59]   Epoch 100 | Train 0.0090 | Val 0.0090 | R2 0.983 | LR 0.0e+00
[02:20:59] Pre-training complete. R2 = 0.983
[02:20:59] Learned relation weights (conv1): {'edge_index_plv': '0.771', 'edge_index_mvar': '0.044', 'edge_index_sc': '0.185'}
[02:20:59] Learned relation weights (conv2): {'edge_index_plv': '0.363', 'edge_index_mvar': '0.305', 'edge_index_sc': '0.333'}


# 10 — HC Data + Holdout + Augmentation

In [10]:
# =============================================================================
# S10 --- HC DATA + SPLIT
# =============================================================================

log.info('=' * 70)
log.info('STAGE 2: HC DATA LOADING')
log.info('=' * 70)

hc_graphs, hc_file_info = load_hc_graphs(
    roi_meta, network_assignment, N_NETWORKS, sc_matrix,
    include_mvar=True
)
hc_train_graphs, hc_test_graphs = split_hc_by_subject(hc_graphs, hc_file_info)

# ── HC holdout for false positive rate estimation ─────────────────────────
# Hold out ~15% of HC subjects (never seen during GVAE training).
# Their anomaly scores provide an unbiased false positive rate estimate.
from sklearn.model_selection import train_test_split

n_holdout = max(1, int(len(hc_train_graphs) * HC_HOLDOUT_FRAC))
hc_train_graphs, hc_holdout_graphs = train_test_split(
    hc_train_graphs, test_size=HC_HOLDOUT_FRAC, random_state=SEED
)
log.info('HC split: %d train, %d test (within-subject), %d holdout (unseen subjects)',
         len(hc_train_graphs), len(hc_test_graphs), len(hc_holdout_graphs))
# ── Derived feature augmentation (v6: expand reconstruction targets) ──────
# Add 4 connectivity-derived features per node to each graph:
#   s_plv:       PLV node strength (sum of PLV edge weights per node)
#   s_mvar_in:   MVAR in-strength (sum of incoming MVAR weights)
#   s_mvar_out:  MVAR out-strength (sum of outgoing MVAR weights)
#   plv_within:  Within-network mean PLV (avg PLV to same-Yeo-network nodes)
#
# These are stored as data.recon_target = [a, omega, chisq, s_plv, s_mvar_in, s_mvar_out, plv_within]
# The encoder input (data.x) is unchanged — still 11 features.

def augment_graph_with_derived_features(graph, network_assignment, n_rois=216):
    """Add derived connectivity features as reconstruction targets."""
    physics = graph.x[:, :3]  # a, omega, chisq
    n_nodes = graph.x.size(0)

    def _get_ea_1d(graph, rel, n_edges):
        """Get edge attributes as a flat 1D tensor."""
        ea = getattr(graph, f'edge_attr_{rel}', None)
        if ea is None:
            return torch.ones(n_edges)
        ea = ea.float().view(-1)  # flatten [n_edges, 1] -> [n_edges]
        if ea.shape[0] != n_edges:
            return torch.ones(n_edges)
        return ea

    # PLV node strength
    s_plv = torch.zeros(n_nodes)
    if hasattr(graph, 'edge_index_plv') and graph.edge_index_plv.numel() > 0:
        ei = graph.edge_index_plv
        ea = _get_ea_1d(graph, 'plv', ei.size(1))
        s_plv.scatter_add_(0, ei[0], ea)
        s_plv = s_plv / max(n_nodes, 1)

    # MVAR in/out strength
    s_mvar_in = torch.zeros(n_nodes)
    s_mvar_out = torch.zeros(n_nodes)
    if hasattr(graph, 'edge_index_mvar') and graph.edge_index_mvar.numel() > 0:
        ei = graph.edge_index_mvar
        ea = _get_ea_1d(graph, 'mvar', ei.size(1)).abs()
        s_mvar_in.scatter_add_(0, ei[1], ea)
        s_mvar_out.scatter_add_(0, ei[0], ea)
        s_mvar_in = s_mvar_in / max(n_nodes, 1)
        s_mvar_out = s_mvar_out / max(n_nodes, 1)

    # Within-network PLV
    plv_within = torch.zeros(n_nodes)
    if hasattr(graph, 'edge_index_plv') and graph.edge_index_plv.numel() > 0:
        ei = graph.edge_index_plv
        ea = _get_ea_1d(graph, 'plv', ei.size(1))
        net_assign = network_assignment[:n_nodes]
        same_net = (net_assign[ei[0]] == net_assign[ei[1]])
        within_sum = torch.zeros(n_nodes)
        within_count = torch.zeros(n_nodes)
        if same_net.any():
            within_sum.scatter_add_(0, ei[0][same_net], ea[same_net])
            within_count.scatter_add_(0, ei[0][same_net], torch.ones(same_net.sum()))
        plv_within = within_sum / within_count.clamp(min=1)

    # Stack: [a, omega, chisq, s_plv, s_mvar_in, s_mvar_out, plv_within]
    derived = torch.stack([s_plv, s_mvar_in, s_mvar_out, plv_within], dim=1)
    graph.recon_target = torch.cat([physics, derived], dim=1)
    return graph

# Augment all graphs
for g in hc_train_graphs:
    augment_graph_with_derived_features(g, network_assignment)
for g in hc_test_graphs:
    augment_graph_with_derived_features(g, network_assignment)
for g in hc_holdout_graphs:
    augment_graph_with_derived_features(g, network_assignment)

log.info('Derived features added: %d features per node (recon_target shape: %s)',
         hc_train_graphs[0].recon_target.shape[1], hc_train_graphs[0].recon_target.shape)


[02:20:59] ======================================================================
[02:20:59] STAGE 2: HC DATA LOADING
[02:20:59] ======================================================================
[02:20:59] HC MVAR loaded: 295 matrices
/opt/anaconda3/lib/python3.11/site-packages/rdata/conversion/_conversion.py:900: UserWarning: Missing constructor for R class "tbl_df". The constructor for class "tbl" will be used instead.
  warnings.warn(
/opt/anaconda3/lib/python3.11/site-packages/rdata/conversion/_conversion.py:900: UserWarning: Missing constructor for R class "tbl". The constructor for class "data.frame" will be used instead.
  warnings.warn(
/opt/anaconda3/lib/python3.11/site-packages/rdata/conversion/_conversion.py:900: UserWarning: Missing constructor for R class "tbl_df". The constructor for class "tbl" will be used instead.
  warnings.warn(
/opt/anaconda3/lib/python3.11/site-packages/rdata/conversion/_conversion.py:900: UserWarning: Missing constructor for R class "tbl". Th

# 11 — MDD Graphs + Augmentation

In [11]:
# =============================================================================
# S11 --- MDD GRAPHS (with LS_E4209 rest2 exclusion)
# =============================================================================

log.info('=' * 70)
log.info('MDD EMPIRICAL GRAPH ASSEMBLY')
log.info('=' * 70)

empirical_graphs, subjects_list, groups_map = build_empirical_graphs(
    ukf_df, roi_meta, plv_all, mvar_all, sc_matrix
)

# Exclude LS_E4209 rest2: this session produces reconstruction errors that
# overflow float64 (anomaly ~9261), indicating corrupted input data.
# Identified in v3b and confirmed across v4 runs.
EXCLUDED_SESSIONS = [('LS_E4209', 'rest2')]
for exc_key in EXCLUDED_SESSIONS:
    if exc_key in empirical_graphs:
        del empirical_graphs[exc_key]
        log.info('EXCLUDED: %s %s (known corrupted signal)', exc_key[0], exc_key[1])

log.info('MDD graphs after exclusion: %d', len(empirical_graphs))

# Augment MDD graphs with derived features
for key, g in empirical_graphs.items():
    augment_graph_with_derived_features(g, network_assignment)
log.info('MDD graphs augmented with derived features')


[02:22:08] ======================================================================
[02:22:08] MDD EMPIRICAL GRAPH ASSEMBLY
[02:22:08] ======================================================================
[02:22:09] MDD graphs: 38 (subjects: 19, groups: {'sham': 8, 'active': 11})
[02:22:09] EXCLUDED: LS_E4209 rest2 (known corrupted signal)
[02:22:09] MDD graphs after exclusion: 37
[02:22:09] MDD graphs augmented with derived features


# 12 — Sweep Function

Trains a GAE for a given `latent_dim`, scores all subjects, returns metrics.
Uses the same training loop as main_analysis_v6 cell 24.

In [12]:
# =============================================================================
# Core sweep function — replicates main_analysis_v6 cell 24 + cell 26 + cell 28
# =============================================================================
import time
from scipy.stats import hypergeom

EDGE_RELS = ['plv', 'sc', 'mvar']
circuit_mask = roi_meta['is_depression_circuit'].values.astype(bool)

def run_single_dim(latent_dim):
    t0 = time.time()
    seed_everything(SEED)

    # ── Build model ──
    gae = HopfGAE(
        encoder_model=encoder,
        latent_dim=latent_dim,
        n_features_out=N_FEATURES_OUT,
        noise_sigma=NOISE_SIGMA,
        z_dropout=Z_DROPOUT,
    )
    n_trainable = sum(p.numel() for p in gae.parameters() if p.requires_grad)
    log.info('  latent_dim=%d, trainable=%d', latent_dim, n_trainable)

    # ── Train (same as main_analysis_v6 cell 24) ──
    criterion = GAELoss(
        lambda_edge=LAMBDA_EDGE,
        lambda_graph=LAMBDA_GRAPH,
        feature_weights=RECON_FEATURE_WEIGHTS,
    )
    trainable_params = [p for p in gae.parameters() if p.requires_grad]
    optimizer = torch.optim.AdamW(trainable_params, lr=GAE_LR, weight_decay=GAE_WD)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=GAE_EPOCHS)
    train_loader = PyGDataLoader(hc_train_graphs, batch_size=GAE_BS, shuffle=True)

    for epoch in range(1, GAE_EPOCHS + 1):
        gae.train()
        for batch_data in train_loader:
            optimizer.zero_grad()
            result = gae(batch_data)
            loss, _ = criterion(result, batch_data, gae_model=gae)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(trainable_params, max_norm=1.0)
            optimizer.step()
        scheduler.step()

    log.info('  Training complete (%.1f min)', (time.time() - t0) / 60)

    # ── Scoring function ──
    gae.eval()
    feat_w = RECON_FEATURE_WEIGHTS.numpy()

    def score_one(g):
        with torch.no_grad():
            result = gae(g)
        target = g.recon_target.numpy()
        recon = result['node_recon'].numpy()
        diff = np.clip(target - recon, -1e3, 1e3)
        node_err = np.average(diff ** 2, axis=1, weights=feat_w)
        return node_err.mean(), node_err

    # ── Score all subjects ──
    hc_test_s = np.array([score_one(g)[0] for g in hc_test_graphs])
    hc_holdout_s = np.array([score_one(g)[0] for g in hc_holdout_graphs])

    mdd_scores = {}
    mdd_roi_scores = {}
    for (subj, sess), g in empirical_graphs.items():
        ms, mr = score_one(g)
        mdd_scores[(subj, sess)] = ms
        mdd_roi_scores[(subj, sess)] = mr

    mdd_r1 = np.array([mdd_scores[(s, 'rest1')]
                        for s in subjects_list if (s, 'rest1') in mdd_scores])

    # ── HC-MDD separation ──
    d_sep = (mdd_r1.mean() - hc_test_s.mean()) / np.sqrt(
        (mdd_r1.var() + hc_test_s.var()) / 2)
    _, p_sep = sp_stats.ttest_ind(mdd_r1, hc_test_s, equal_var=False)

    d_holdout = (mdd_r1.mean() - hc_holdout_s.mean()) / np.sqrt(
        (mdd_r1.var() + hc_holdout_s.var()) / 2)

    # ── Intervention effects ──
    active_subs = [s for s in subjects_list
                   if groups_map.get(s, '').lower() in ('active', 'experimental')]

    def intervention_d(roi_subset=None):
        delta_act, delta_sha = [], []
        for s in subjects_list:
            if (s, 'rest1') not in mdd_roi_scores or (s, 'rest2') not in mdd_roi_scores:
                continue
            r1 = mdd_roi_scores[(s, 'rest1')]
            r2 = mdd_roi_scores[(s, 'rest2')]
            if roi_subset is not None:
                r1, r2 = r1[roi_subset], r2[roi_subset]
            delta = r2.mean() - r1.mean()
            if s in active_subs:
                delta_act.append(delta)
            else:
                delta_sha.append(delta)
        if len(delta_act) < 2 or len(delta_sha) < 2:
            return 0.0, 1.0, 1.0
        a, sh = np.array(delta_act), np.array(delta_sha)
        pooled = np.sqrt((a.var() + sh.var()) / 2)
        d = (a.mean() - sh.mean()) / max(pooled, 1e-12)
        _, p = sp_stats.ttest_ind(a, sh, equal_var=False)
        rng = np.random.RandomState(SEED)
        all_d = np.concatenate([a, sh])
        n_a = len(a)
        perm_ds = np.zeros(5000)
        for ip in range(5000):
            idx = rng.permutation(len(all_d))
            g1, g2 = all_d[idx[:n_a]], all_d[idx[n_a:]]
            ps = np.sqrt((g1.var() + g2.var()) / 2)
            perm_ds[ip] = (g1.mean() - g2.mean()) / max(ps, 1e-12)
        perm_p = (np.abs(perm_ds) >= np.abs(d)).mean()
        return d, p, perm_p

    wb_d, wb_p, wb_pp = intervention_d(None)
    circ_d, circ_p, circ_pp = intervention_d(circuit_mask)
    lim_mask = roi_meta['network'].values == 'Limbic'
    lim_d, lim_p, lim_pp = intervention_d(lim_mask)
    sub_mask = roi_meta['network'].values == 'Subcortical'
    sub_d, sub_p, sub_pp = intervention_d(sub_mask)

    # ── Circuit enrichment ──
    mean_roi = np.zeros(len(roi_meta))
    count_roi = np.zeros(len(roi_meta))
    for (subj, sess), roi_s in mdd_roi_scores.items():
        if sess == 'rest1':
            mean_roi += roi_s
            count_roi += 1
    mean_roi = mean_roi / np.maximum(count_roi, 1)
    ranking = np.argsort(-mean_roi)
    base_rate = circuit_mask.mean()

    enrich, hyper_p = {}, {}
    N_total, K_circ = len(roi_meta), int(circuit_mask.sum())
    for n in [10, 15, 20, 30]:
        nc = int(circuit_mask[ranking[:n]].sum())
        enrich[n] = (nc / n) / base_rate
        hyper_p[n] = hypergeom.sf(nc - 1, N_total, K_circ, n)

    elapsed = (time.time() - t0) / 60
    log.info('  d_sep=%+.3f  wb_d=%+.3f(pp=%.4f)  sub_d=%+.3f(pp=%.4f)  '
             'E10=%.2fx  %.1fmin',
             d_sep, wb_d, wb_pp, sub_d, sub_pp, enrich[10], elapsed)

    return {
        'latent_dim': latent_dim, 'n_trainable': n_trainable,
        'd_sep': d_sep, 'p_sep': p_sep, 'd_holdout': d_holdout,
        'wb_d': wb_d, 'wb_p': wb_p, 'wb_pp': wb_pp,
        'circ_d': circ_d, 'circ_p': circ_p, 'circ_pp': circ_pp,
        'lim_d': lim_d, 'lim_p': lim_p, 'lim_pp': lim_pp,
        'sub_d': sub_d, 'sub_p': sub_p, 'sub_pp': sub_pp,
        'enrich_10': enrich[10], 'enrich_15': enrich[15],
        'enrich_20': enrich[20], 'enrich_30': enrich[30],
        'hyper_p_10': hyper_p[10], 'hyper_p_20': hyper_p[20],
        'elapsed_min': elapsed,
    }

log.info('Sweep function ready.')

[02:22:09] Sweep function ready.


# 13 — Run Sweep

In [13]:
# =============================================================================
# Execute sweep
# =============================================================================

results = []
for dim in SWEEP_DIMS:
    log.info('=' * 60)
    log.info('SWEEP: latent_dim = %d', dim)
    log.info('=' * 60)
    r = run_single_dim(dim)
    results.append(r)

df = pd.DataFrame(results)
df.to_csv(RESULTS_V4 / 'bottleneck_sweep.csv', index=False)
log.info('Sweep complete. Saved to %s', RESULTS_V4 / 'bottleneck_sweep.csv')

[02:22:09] ============================================================
[02:22:09] SWEEP: latent_dim = 3
[02:22:09] ============================================================
[02:22:09]   latent_dim=3, trainable=1762
[02:26:19]   Training complete (4.2 min)
[02:26:20]   d_sep=+3.679  wb_d=+1.362(pp=0.0198)  sub_d=+1.788(pp=0.0020)  E10=2.50x  4.2min
[02:26:20] ============================================================
[02:26:20] SWEEP: latent_dim = 4
[02:26:20] ============================================================
[02:26:20]   latent_dim=4, trainable=1802
[02:30:33]   Training complete (4.2 min)
[02:30:33]   d_sep=+3.493  wb_d=+1.363(pp=0.0182)  sub_d=+1.802(pp=0.0022)  E10=2.50x  4.2min
[02:30:33] ============================================================
[02:30:33] SWEEP: latent_dim = 6
[02:30:33] ============================================================
[02:30:33]   latent_dim=6, trainable=1882
[02:34:48]   Training complete (4.2 min)
[02:34:49]   d_sep=+3.538  wb_d=

# 14 — Results Table

In [14]:
# =============================================================================
# Summary table
# =============================================================================

print('=' * 110)
print('BOTTLENECK DIMENSION SWEEP')
print('=' * 110)
print(f"{'d_z':>4s} {'params':>7s} | {'HC-MDD d':>9s} {'holdout d':>10s} | "
      f"{'WB d':>6s} {'WB pp':>7s} {'Sub d':>6s} {'Sub pp':>7s} {'Lim d':>6s} {'Lim pp':>7s} | "
      f"{'E10':>5s} {'E15':>5s} {'E20':>5s} {'hp10':>6s}")
print('-' * 110)

for _, r in df.iterrows():
    d = int(r['latent_dim'])
    marker = ' <' if d == 8 else ''
    print(f"{d:4d} {int(r['n_trainable']):7d} | "
          f"{r['d_sep']:+9.3f} {r['d_holdout']:+10.3f} | "
          f"{r['wb_d']:+6.3f} {r['wb_pp']:7.4f} "
          f"{r['sub_d']:+6.3f} {r['sub_pp']:7.4f} "
          f"{r['lim_d']:+6.3f} {r['lim_pp']:7.4f} | "
          f"{r['enrich_10']:5.2f} {r['enrich_15']:5.2f} {r['enrich_20']:5.2f} "
          f"{r['hyper_p_10']:6.4f}{marker}")

print('-' * 110)
print('< = main analysis dimension (d_z=8)')

all_wb_sig = all(r['wb_pp'] < 0.05 for _, r in df.iterrows())
all_sub_sig = all(r['sub_pp'] < 0.05 for _, r in df.iterrows())
all_enrich_sig = all(r['hyper_p_10'] < 0.05 for _, r in df.iterrows())

print()
print(f"Whole-brain intervention perm p<0.05 for ALL dims: {all_wb_sig}")
print(f"Subcortical intervention perm p<0.05 for ALL dims: {all_sub_sig}")
print(f"Top-10 enrichment hypergeom p<0.05 for ALL dims: {all_enrich_sig}")
if all_wb_sig and all_sub_sig and all_enrich_sig:
    print("CONCLUSION: Findings are robust to bottleneck dimension.")

BOTTLENECK DIMENSION SWEEP
 d_z  params |  HC-MDD d  holdout d |   WB d   WB pp  Sub d  Sub pp  Lim d  Lim pp |   E10   E15   E20   hp10
--------------------------------------------------------------------------------------------------------------
   3    1762 |    +3.679     +2.827 | +1.362  0.0198 +1.788  0.0020 +1.616  0.0058 |  2.50  2.30  2.19 0.0020
   4    1802 |    +3.493     +2.692 | +1.363  0.0182 +1.802  0.0022 +1.459  0.0098 |  2.50  2.09  2.03 0.0020
   6    1882 |    +3.538     +2.713 | +1.261  0.0312 +1.747  0.0040 +1.449  0.0114 |  2.19  2.09  2.03 0.0133
   8    1962 |    +3.442     +2.679 | +1.304  0.0260 +1.746  0.0046 +1.414  0.0128 |  1.88  1.67  1.72 0.0587 <
  12    2122 |    +3.565     +2.773 | +1.291  0.0270 +1.695  0.0056 +1.427  0.0126 |  1.88  1.67  1.72 0.0587
--------------------------------------------------------------------------------------------------------------
< = main analysis dimension (d_z=8)

Whole-brain intervention perm p<0.05 for ALL dims: T

# 15 — Sweep Visualization

In [15]:
# =============================================================================
# Multi-panel figure
# =============================================================================
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 4, figsize=(18, 4.5))
dims = df['latent_dim'].values

ax = axes[0]
ax.bar(range(len(dims)), df['d_sep'].values, color='#2196F3', alpha=0.8)
ax.set_xticks(range(len(dims))); ax.set_xticklabels(dims)
ax.set_xlabel('$d_z$'); ax.set_ylabel("Cohen's $d$")
ax.set_title('(a) HC-MDD Separation', fontweight='bold', fontsize=11)

ax = axes[1]
x = np.arange(len(dims))
w = 0.2
for j, (key, label, color) in enumerate([
    ('wb_d', 'Whole-brain', '#4CAF50'), ('sub_d', 'Subcortical', '#FF9800'),
    ('lim_d', 'Limbic', '#E91E63'), ('circ_d', 'Circuit', '#9C27B0')]):
    ax.bar(x + j*w - 1.5*w, df[key].values, w, label=label, color=color, alpha=0.8)
ax.set_xticks(x); ax.set_xticklabels(dims)
ax.set_xlabel('$d_z$'); ax.set_ylabel("Cohen's $d$")
ax.set_title('(b) Intervention Effects', fontweight='bold', fontsize=11)
ax.legend(fontsize=8)

ax = axes[2]
for n, marker, color in [(10,'o','#E91E63'), (15,'s','#9C27B0'),
                          (20,'D','#3F51B5'), (30,'^','#607D8B')]:
    ax.plot(dims, df[f'enrich_{n}'].values, f'-{marker}', label=f'Top-{n}',
            color=color, markersize=6)
ax.axhline(1.0, color='grey', ls='--', lw=0.8, label='Chance')
ax.set_xlabel('$d_z$'); ax.set_ylabel('Enrichment ratio')
ax.set_title('(c) Circuit Enrichment', fontweight='bold', fontsize=11)
ax.legend(fontsize=8)

ax = axes[3]
for key, label, color in [('wb_pp','Whole-brain','#4CAF50'),
                           ('sub_pp','Subcortical','#FF9800'),
                           ('lim_pp','Limbic','#E91E63')]:
    ax.plot(dims, df[key].values, '-o', label=label, color=color, markersize=6)
ax.axhline(0.05, color='red', ls='--', lw=0.8, alpha=0.5, label='p=0.05')
ax.set_xlabel('$d_z$'); ax.set_ylabel('Permutation $p$-value')
ax.set_title('(d) Intervention Significance', fontweight='bold', fontsize=11)
ax.legend(fontsize=8); ax.set_yscale('log')

plt.tight_layout()
Path('img').mkdir(exist_ok=True)
plt.savefig('img/fig_bottleneck_sweep.png', dpi=300, bbox_inches='tight')
plt.savefig('img/fig_bottleneck_sweep.pdf', bbox_inches='tight')
plt.show()
log.info('-> img/fig_bottleneck_sweep.png + .pdf')

/var/folders/pr/q2gt8qfd2px6jqkj3yd3kzs80000gn/T/ipykernel_41097/739956423.py:53: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
[02:43:20] -> img/fig_bottleneck_sweep.png + .pdf
